In [0]:
from pyspark.sql import functions as F

#Criando o schema da camada Gold
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.gold")

#Lendo os metadados tratados e criando uma chave substituta para cada filme
#A dimensão tem uma linha por filme, a chave substituta é usada para ligar a fato e as tabelas-ponte
df_dim_movies = (
    spark.table("workspace.silver.tb_info_filmes")
    .withColumn("sk_movie_id", F.monotonically_increasing_id())
    .select(
        "sk_movie_id",
        "id_filme",
        "titulo",
        "data_lancamento",
        "ano_lancamento",
        "duracao_minutos",
        "idioma_original",
        "status_filme",
        "sinopse"))

#Salvando a dimensão de filmes na Gold
df_dim_movies.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_movies")

display(df_dim_movies.limit(10))

In [0]:
#Lendo as tabelas tratadas da Silver
df_financeiro = spark.table("workspace.silver.tb_financeiro_filmes")
df_metricas = spark.table("workspace.silver.tb_metricas_engajamento")

#Considerei somente filmes com status lançado. A fato considera somente filmes lançados, conforme o grão definido no exercício
df_filmes_lancados = (
    spark.table("workspace.gold.dim_movies")
    .filter(F.col("status_filme") == "Lançado")
    .select("sk_movie_id", "id_filme"))

#Unindo os dados mantendo uma linha por filme
#Utilizo left join para manter todos os filmes lançados, inclusive os que não têm alguma métrica ou informação financeira
df_fact_movies = (
    df_filmes_lancados
#Removo repetição por chave antes dos joins para não multiplicar linhas da fato
    .join(df_financeiro.dropDuplicates(["id_filme"]), on="id_filme", how="left")
    .join(df_metricas.dropDuplicates(["id_filme"]), on="id_filme", how="left")
    .select(
        "sk_movie_id",
        F.col("orcamento_usd").cast("decimal(18,2)").alias("orcamento_usd"),
        F.col("receita_usd").cast("decimal(18,2)").alias("receita_usd"),
        F.col("lucro_usd").cast("decimal(18,2)").alias("lucro_usd"),
        F.col("orcamento_brl").cast("decimal(18,2)").alias("orcamento_brl"),
        F.col("receita_brl").cast("decimal(18,2)").alias("receita_brl"),
        F.col("lucro_brl").cast("decimal(18,2)").alias("lucro_brl"),
        "popularidade",
        "nota_media_tmdb",
        "qtd_votos_tmdb",
        "nota_media_imdb",
        "qtd_votos_imdb"))

display(df_fact_movies.limit(10))

In [0]:
display(
    df_filmes_lancados
    .join(df_financeiro, on="id_filme", how="left")
    .select("id_filme", "orcamento_usd", "receita_usd")
    .limit(10))

In [0]:
#Salvando as métricas consolidadas dos filmes lançados na Gold
df_fact_movies.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.fact_movies_performance")

display(spark.table("workspace.gold.fact_movies_performance").limit(10))

In [0]:
from pyspark.sql.window import Window

#Criando as chaves de gênero em uma ordem fixa para manter os relacionamentos consistentes
janela_generos = Window.orderBy("nome_genero")

df_dim_genres = (
    spark.table("workspace.silver.tb_generos")
    .select("nome_genero")
    .distinct()
    .withColumn("sk_genre_id", F.row_number().over(janela_generos).cast("long"))
    .select("sk_genre_id", "nome_genero"))

df_dim_genres.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_genres")


#Usando a dimensão salva para garantir que a ponte tenha as mesmas chaves
df_bridge_movie_genre = (
    spark.table("workspace.silver.tb_generos")
    .join(
        spark.table("workspace.gold.dim_movies").select("sk_movie_id", "id_filme"),
        on="id_filme",
        how="inner"
    )
    .join(
        spark.table("workspace.gold.dim_genres"),
        on="nome_genero",
        how="inner"
    )
    .select("sk_movie_id", "sk_genre_id")
    .dropDuplicates())

df_bridge_movie_genre.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.bridge_movie_genre")

In [0]:
entidades = spark.table("workspace.silver.tb_pessoas_empresas")
filmes = spark.table("workspace.gold.dim_movies").select("sk_movie_id", "id_filme")

#Guardo o tipo de atuação junto ao nome para diferenciar, por exemplo, ator e diretor nas consultas
#Criando a dimensão de pessoas
df_dim_people = (
    entidades
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .distinct()
    .withColumn("sk_person_id", F.monotonically_increasing_id())
    .select("sk_person_id", "nome_pessoa", "tipo_pessoa"))

df_dim_people.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_people")

#Ligando os filmes às pessoas usando a dimensão já salva
df_bridge_movie_person = (
    entidades
    .filter(F.col("tipo_entidade").isin("Ator", "Diretor", "Roteirista"))
    .select(
        "id_filme",
        F.col("nome_entidade").alias("nome_pessoa"),
        F.col("tipo_entidade").alias("tipo_pessoa")
    )
    .join(filmes, on="id_filme", how="inner")
    .join(
        spark.table("workspace.gold.dim_people"),
        on=["nome_pessoa", "tipo_pessoa"],
        how="inner"
    )
    .select("sk_movie_id", "sk_person_id")
    .dropDuplicates())

df_bridge_movie_person.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.bridge_movie_person")


#Criando a dimensão de produtoras
df_dim_companies = (
    entidades
    .filter(F.col("tipo_entidade") == "Produtora")
    .select(F.col("nome_entidade").alias("nome_produtora"))
    .distinct()
    .withColumn("sk_company_id", F.monotonically_increasing_id())
    .select("sk_company_id", "nome_produtora"))

df_dim_companies.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_companies")

#Ligando os filmes às produtoras usando a dimensão já salva
df_bridge_movie_company = (
    entidades
    .filter(F.col("tipo_entidade") == "Produtora")
    .select("id_filme", F.col("nome_entidade").alias("nome_produtora"))
    .join(filmes, on="id_filme", how="inner")
    .join(
        spark.table("workspace.gold.dim_companies"),
        on="nome_produtora",
        how="inner"
    )
    .select("sk_movie_id", "sk_company_id")
    .dropDuplicates())

df_bridge_movie_company.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.bridge_movie_company")

In [0]:
#Resumindo as avaliações dos usuários por filme
df_reviews_resumo = (
    spark.table("workspace.silver.tb_avaliacoes_usuarios")
    .groupBy("id_filme")
    .agg(
        F.count("*").cast("int").alias("qtd_avaliacoes_usuarios"),
        F.round(F.avg("nota_usuario"), 2).alias("nota_media_usuarios")))

#Relacionando o resumo à chave do filme e criando a chave da avaliação resumida
df_dim_reviews = (
    df_reviews_resumo
    .join(
        spark.table("workspace.gold.dim_movies").select("sk_movie_id", "id_filme"),
        on="id_filme",
        how="inner"
    )
    .withColumn("sk_review_id", F.monotonically_increasing_id())
    .select(
        "sk_review_id",
        "sk_movie_id",
        "qtd_avaliacoes_usuarios",
        "nota_media_usuarios"))

df_dim_reviews.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.gold.dim_reviews")

display(df_dim_reviews.limit(10))

In [0]:
spark.sql("SHOW TABLES IN workspace.gold").display()